# 🎬 Lip-Sync con Wav2Lip
**Instrucciones:**
1. Ve a `Runtime → Change runtime type → T4 GPU` (gratis)
2. Ejecuta las celdas **en orden** con el botón ▶️
3. En la celda de **Subir archivos**, sube tu video y tu audio
4. Al final descarga el video sincronizado

> ⏱️ Un video de ~1 minuto tarda aprox. 3-6 minutos con GPU T4

## Paso 1 — Instalar dependencias

In [ ]:
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip
!pip install -q librosa==0.9.2 numpy==1.23.5 opencv-python tqdm
print('✅ Dependencias instaladas')

## Paso 2 — Descargar modelo preentrenado

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)

# Descarga el modelo wav2lip_gan (mejor calidad visual)
!wget -q --show-progress -O checkpoints/wav2lip_gan.pth \
  'https://iiitaphyd-my.sharepoint.com/:u:/g/personal/radrabha_m_research_iiit_ac_in/EdjI7bZlgApMqsVoEUUXpLsBxqXbn5z8VTmoxp55YNDcIA?e=n9ljGW&download=1' \
  || echo '⚠️ Falló el mirror 1, intentando mirror 2...'

# Si el anterior falla, usa este mirror de HuggingFace
if not os.path.exists('checkpoints/wav2lip_gan.pth') or os.path.getsize('checkpoints/wav2lip_gan.pth') < 1000000:
    !wget -q --show-progress -O checkpoints/wav2lip_gan.pth \
      'https://huggingface.co/numz/wav2lip_studio/resolve/main/Wav2Lip/wav2lip_gan.pth'

size = os.path.getsize('checkpoints/wav2lip_gan.pth') / 1e6
print(f'✅ Modelo descargado: {size:.1f} MB')

## Paso 3 — Subir tu video y audio
Sube:
- **Video**: el avatar de HeyGen (`.mp4`)
- **Audio**: el audio que quieres sincronizar (`.mp3` o `.wav`)

In [ ]:
from google.colab import files
import shutil, os

print('📁 Sube tu VIDEO (mp4):')
uploaded_video = files.upload()
video_filename = list(uploaded_video.keys())[0]
shutil.copy(video_filename, f'/content/Wav2Lip/input_video.mp4')
print(f'✅ Video subido: {video_filename}')

print('\n🎵 Sube tu AUDIO (mp3 o wav):')
uploaded_audio = files.upload()
audio_filename = list(uploaded_audio.keys())[0]
ext = os.path.splitext(audio_filename)[1]
shutil.copy(audio_filename, f'/content/Wav2Lip/input_audio{ext}')
print(f'✅ Audio subido: {audio_filename}')

AUDIO_PATH = f'/content/Wav2Lip/input_audio{ext}'

## Paso 4 — Ejecutar Lip-Sync 🚀

In [ ]:
import os

VIDEO_PATH = '/content/Wav2Lip/input_video.mp4'
OUTPUT_PATH = '/content/Wav2Lip/results/result_voice.mp4'

os.makedirs('/content/Wav2Lip/results', exist_ok=True)

print('🔄 Procesando lip-sync... (puede tardar 3-8 minutos)')

!python /content/Wav2Lip/inference.py \
    --checkpoint_path /content/Wav2Lip/checkpoints/wav2lip_gan.pth \
    --face "{VIDEO_PATH}" \
    --audio "{AUDIO_PATH}" \
    --outfile "{OUTPUT_PATH}" \
    --pads 0 10 0 0 \
    --resize_factor 1

if os.path.exists(OUTPUT_PATH):
    size = os.path.getsize(OUTPUT_PATH) / 1e6
    print(f'\n✅ ¡Listo! Video generado: {size:.1f} MB')
else:
    print('❌ Algo salió mal, revisa los errores arriba')

## Paso 5 — Ver preview y descargar

In [ ]:
# Preview en el notebook
from IPython.display import HTML
from base64 import b64encode

OUTPUT_PATH = '/content/Wav2Lip/results/result_voice.mp4'

with open(OUTPUT_PATH, 'rb') as f:
    video_data = f.read()

video_b64 = b64encode(video_data).decode()
HTML(f'''
<video width="640" controls>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
''')

In [ ]:
# Descargar el video final
from google.colab import files
files.download('/content/Wav2Lip/results/result_voice.mp4')
print('⬇️ Descarga iniciada')

---
## ⚙️ Parámetros opcionales para ajustar calidad

Si el resultado no te convence, modifica estos valores en el **Paso 4**:

| Parámetro | Descripción | Valores sugeridos |
|---|---|---|
| `--pads 0 10 0 0` | Padding alrededor de la boca (top, bottom, left, right) | Aumenta el bottom (ej: `0 20 0 0`) si la boca se corta |
| `--resize_factor 1` | Reducir resolución para procesar más rápido | `1` = original, `2` = mitad |
| `--nosmooth` | Desactiva suavizado temporal | Agrega este flag si los labios se ven borrosos |

### Modelo alternativo (más preciso, más lento)
Cambia `wav2lip_gan.pth` por `wav2lip.pth` para mejor sincronización pero imagen menos nítida.